In [0]:
#gold table creation.
from pyspark.sql.functions import col, sum, round, month, year

df_s = spark.readStream \
          .table("expense_silver")


df_gold = df_s.withColumn("month", month(col("expense_dt")))\
              .withColumn("year", year(col("expense_dt")))\
              .groupBy("year","month","Category") \
              .agg(round(sum("Amount").cast("decimal(18,2)"),2).alias("expense")) \
              .withColumnRenamed("Category", "custom_category") 

query = df_gold.writeStream\
                .format("delta")\
                .outputMode("complete")\
                .option("checkpointLocation", "/Volumes/exp_catalog/exp_schema/exp_volume/exp_gold")\
                .trigger(availableNow=True)\
                .toTable("expense_gold")

query.awaitTermination()

gold_display = spark.table("expense_gold")
gold_display.printSchema()
display(gold_display)